In [1]:
!pip install -q unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.3/82.3 MB 21.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 44.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 100.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 90.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 90.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 8.8 MB/s eta 0:00:00
   ━━━━

In [2]:
from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    "Arup330/Abdomen_CoT_llama_lora",
    load_in_4bit = True,  # Use 4bit to reduce memory use. False for 16bit LoRA.
    use_gradient_checkpointing = "unsloth",  # True or "unsloth" for long context
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.2: Fast Mllama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
total weights      : 7.231 GiB
no_split classes   : ['MllamaCrossAttentionDecoderLayer', 'MllamaSelfAttentionDecoderLayer', 'MllamaVisionEncoderLayer']
output head        : lm_head -> cuda:1
head headroom      : 0.342 GiB
activation reserve : 9.188 GiB requested
  cuda:0  budget  12.95 GiB  weights  3.760 GiB  free  9.192 GiB  reserve  9.188 GiB
  cuda:1  budget  13.00 GiB  weights  3.471 GiB  free  9.525 Gi

Loading weights:   0%|          | 0/906 [00:00<?, ?it/s]

In [3]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers

    r = 16,           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    # target_modules = "all-linear", # Optional now! Can specify a list if needed
)

RuntimeError: Unsloth: You already added LoRA adapters to your model!

In [4]:
#BrainFace_ROOT = "/kaggle/input/datasets/arups330/slackdataset/SLAKE_split_en_realimages/Brain_Face"
#NECK_ROOT  = "/kaggle/input/datasets/arups330/slackdataset/SLAKE_split_en_realimages/Neck"
ABDOMEN_ROOT = "/kaggle/input/datasets/arups330/slackdataset/SLAKE_split_en_realimages/Abdomen/CT"
import os
for root, dirs, files in os.walk("/kaggle/input"):
    if root.endswith("Abdomen/CT"):
        print(root)

/kaggle/input/datasets/arups330/slackdataset/SLAKE_split_en_realimages/Abdomen/CT


In [5]:
SPLIT = "test"       # you said "abdomen test data" -- change to "train" if needed
CSV_TYPE = "closed"   # <-- run once as "open", then change to "closed" and rerun
# ---------------------------------------------------------------------------
# 1. Find only the CSV_TYPE csvs (open.csv OR closed.csv) across CT test folders
# ---------------------------------------------------------------------------
import os
import glob
import pandas as pd
from PIL import Image

csv_paths = glob.glob(os.path.join(ABDOMEN_ROOT, SPLIT, f"{CSV_TYPE}.csv"))
print(f"Found {len(csv_paths)} {CSV_TYPE} CSVs for split='{SPLIT}':")
for p in csv_paths:
    print(" ", p)

if not csv_paths:
    raise FileNotFoundError(
        f"No {CSV_TYPE}.csv found under {ABDOMEN_ROOT}/{SPLIT}/{CSV_TYPE}.csv -- "
        f"check ABDOMEN_ROOT and SPLIT."
    )

frames = []
for csv_path in csv_paths:
    split_dir = os.path.dirname(csv_path)   # e.g. Abdomen/CT/test
    df = pd.read_csv(csv_path)
    df["split_dir"] = split_dir
    frames.append(df)

data_df = pd.concat(frames, ignore_index=True)
print(f"\nTotal {CSV_TYPE} rows: {len(data_df)}")
print("Columns:", list(data_df.columns))

IMG_COL = "image_file" if "image_file" in data_df.columns else "img_name"
print(f"Using image column: {IMG_COL}")

def resolve_image_path(split_dir: str, img_name: str) -> str:
    flat_name = os.path.basename(img_name)
    candidates = [
        os.path.join(split_dir, img_name),
        os.path.join(split_dir, flat_name),
        os.path.join(split_dir, img_name.replace("/", "_")),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raise FileNotFoundError(f"Could not find image for {img_name!r} in {split_dir}")

Found 1 closed CSVs for split='test':
  /kaggle/input/datasets/arups330/slackdataset/SLAKE_split_en_realimages/Abdomen/CT/test/closed.csv

Total closed rows: 127
Columns: ['image_file', 'img_id', 'location', 'modality', 'question', 'answer', 'q_lang', 'answer_type', 'content_type', 'base_type', 'qid', 'triple', 'split_dir']
Using image column: image_file


In [6]:
# ---------------------------------------------------------------------------
# 2. Your exact CoT instruction template (unchanged)
# ---------------------------------------------------------------------------
INSTRUCTION_TEMPLATE = """Context: You are a senior radiologist performing structured diagnostic reasoning.

Your goal is NOT just to answer, but to produce a **step-by-step clinical reasoning chain (Chain-of-Thought)** grounded in the image.

-----------------------------------
INPUT:
- Question: {question}
- Ground Truth Answer: {answer}
-----------------------------------

TASK INSTRUCTIONS:

You MUST follow a strict multi-step reasoning process:

Step 1: Identify Imaging Modality
- Determine modality (X-ray / CT / MRI / Ultrasound)
- Explain visual clues (contrast, density, grayscale pattern)

Step 2: Global Image Understanding
- Describe anatomical region
- Identify orientation (axial, sagittal, coronal, frontal)

Step 3: Region-wise Analysis
- Divide image into anatomical zones
- Analyze each region systematically

Step 4: Visual Feature Extraction
- Density (hyperdense / hypodense)
- Shape, edges, symmetry
- Texture abnormalities

Step 5: Abnormality Detection
- Identify pathology (if present)
- Localize precisely

Step 6: Clinical Reasoning
- Link findings to medical knowledge
- Explain WHY the abnormality matches the condition

Step 7: Question Understanding
- What exactly is the question asking?
- Type: (yes/no, location, modality, abnormality)

Step 8: Answer Justification
- Justify the provided answer: "{answer}"
- Explain why it is correct based on image evidence

-----------------------------------
OUTPUT FORMAT (STRICT):

1. Imaging Modality:
2. Anatomical Region:
3. Orientation:
4. Region-wise Findings:
5. Key Visual Features:
6. Detected Abnormality:
7. Clinical Interpretation:
8. Question Analysis:
9. Final Answer Justification:

IMPORTANT:
- Do NOT skip steps
- Do NOT give short answers
- Each step must contain 2-4 sentences
- Use medical terminology
"""

def generate_cot(image: Image.Image, question: str, answer: str) -> str:
    prompt_text = INSTRUCTION_TEMPLATE.format(question=question, answer=answer)
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt_text},
                {"type": "image", "image": image},
            ],
        }
    ]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(image, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")

    output_ids = model.generate(
        **inputs,
        max_new_tokens=1024,
        use_cache=True,
        temperature=0.3,
        min_p=0.1,
    )
    return tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()


In [7]:
# ---------------------------------------------------------------------------
# 3. Generate CoT for every row of THIS CSV_TYPE only
# ---------------------------------------------------------------------------
print(f"Generating CoT for {len(data_df)} '{CSV_TYPE}' rows...")

cots = []
for i, row in data_df.iterrows():
    try:
        img_path = resolve_image_path(row["split_dir"], row[IMG_COL])
        image = Image.open(img_path).convert("RGB")
        cot = generate_cot(image, row["question"], row["answer"])
    except Exception as e:
        print(f"  [WARN] row {i} failed ({row.get(IMG_COL)}): {e}")
        cot = ""
    cots.append(cot)

    if i % 5 == 0:
        print(f"  [{i}/{len(data_df)}] done")

data_df["CoT"] = cots
print("\nCoT generation complete.")

Generating CoT for 127 'closed' rows...


[unsloth_zoo.log|WARNING]Unsloth: torch.compile hit one of Unsloth's own `torch.compiler.disable`d gradient-checkpointing hooks inside MllamaPrecomputedAspectRatioEmbedding_forward; running it eagerly from here. Training is unaffected apart from speed.
[unsloth_zoo.log|WARNING]Unsloth: torch.compile hit one of Unsloth's own `torch.compiler.disable`d gradient-checkpointing hooks inside MllamaPrecomputedPositionEmbedding_forward; running it eagerly from here. Training is unaffected apart from speed.


  [0/127] done
  [5/127] done
  [10/127] done
  [15/127] done
  [20/127] done
  [25/127] done
  [30/127] done
  [35/127] done
  [40/127] done
  [45/127] done
  [50/127] done
  [55/127] done
  [60/127] done
  [65/127] done
  [70/127] done
  [75/127] done
  [80/127] done
  [85/127] done
  [90/127] done
  [95/127] done
  [100/127] done
  [105/127] done
  [110/127] done
  [115/127] done
  [120/127] done
  [125/127] done

CoT generation complete.


In [8]:
data_df.head()

,image_file,img_id,location,modality,question,answer,q_lang,answer_type,content_type,base_type,qid,triple,split_dir,CoT
0,xmlab103_source.jpg,103,Abdomen,CT,Does the picture contain liver?,No,en,CLOSED,Organ,vqa,11949,['vhead' '_' '_'],/kaggle/input/datasets/arups330/slackdataset/S...,**Step 1: Imaging Modality**\n\nThe provided i...
1,xmlab103_source.jpg,103,Abdomen,CT,Does the picture contain lung?,Yes,en,CLOSED,Organ,vqa,11950,['vhead' '_' '_'],/kaggle/input/datasets/arups330/slackdataset/S...,**Step 1: Imaging Modality**\n\nThe provided i...
2,xmlab103_source.jpg,103,Abdomen,CT,Is the lung healthy?,No,en,CLOSED,Abnormality,vqa,11952,['vhead' '_' '_'],/kaggle/input/datasets/arups330/slackdataset/S...,**Step 1: Imaging Modality**\n\nThe provided i...
3,xmlab103_source.jpg,103,Abdomen,CT,"Which is the biggest in this image,lung,liver ...",Lung,en,CLOSED,Size,vqa,11954,['vhead' '_' '_'],/kaggle/input/datasets/arups330/slackdataset/S...,**Step 1: Imaging Modality**\n\nThe provided i...
4,xmlab208_source.jpg,208,Abdomen,CT,Does the picture contain liver?,Yes,en,CLOSED,Organ,vqa,12078,['vhead' '_' '_'],/kaggle/input/datasets/arups330/slackdataset/S...,**Step 1: Imaging Modality**\n\nThe provided i...


In [9]:
# ---------------------------------------------------------------------------
# 4. Save as <csv_type>_with_CoT.csv, mirroring the SAME relative folder
#    structure (e.g. CT/test/open_with_CoT.csv) under /kaggle/working -- this
#    mirrors the exact path you'll push into the real dataset folder later.
# ---------------------------------------------------------------------------
OUTPUT_ROOT = "/kaggle/working/Abdomen_with_CoT"

for split_dir, group_df in data_df.groupby("split_dir"):
    rel_dir = os.path.relpath(split_dir, ABDOMEN_ROOT)   # e.g. "CT/test"
    out_dir = os.path.join(OUTPUT_ROOT, rel_dir)
    os.makedirs(out_dir, exist_ok=True)

    out_name = f"{CSV_TYPE}_with_CoT.csv"   # open_with_CoT.csv OR closed_with_CoT.csv
    out_path = os.path.join(out_dir, out_name)
    group_df.drop(columns=["split_dir"]).to_csv(out_path, index=False)
    print(f"Saved: {out_path}  ({len(group_df)} rows)")

print(f"\nDone with '{CSV_TYPE}'. Now change CSV_TYPE to the other value and rerun from the top.")
print(f"Files saved under: {OUTPUT_ROOT} (mirrors Abdomen/<modality>/{SPLIT}/ structure)")

Saved: /kaggle/working/Abdomen_with_CoT/test/closed_with_CoT.csv  (127 rows)

Done with 'closed'. Now change CSV_TYPE to the other value and rerun from the top.
Files saved under: /kaggle/working/Abdomen_with_CoT (mirrors Abdomen/<modality>/test/ structure)
